In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Training on synthetic data

In [ ]:
import sys
import os

sys.path.append(os.path.abspath("..")) 

In [ ]:
import torch


def _patched_solve(B, A):
    return torch.linalg.solve(A, B), None


torch.solve = _patched_solve

from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt

from losses import TimePointOverallLoss
from timepoint import TimePointModel  # your encoder+decoders combined

In [ ]:
SIGNAL_TO_TRAIN = "cbfv"  # or "abp"

In [ ]:
def collate_fn(batch):
    return {
        "x": torch.stack([b["x"] for b in batch]),
        "x_w": torch.stack([b["x_w"] for b in batch]),
        "kp": torch.stack([b["kp"] for b in batch]),
        "kp_w": torch.stack([b["kp_w"] for b in batch]),
        "match_mask": torch.stack([b["match_mask"] for b in batch]),
    }

In [ ]:
from data_loader import NPZLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")


dataset = NPZLoader(
    "../dataset_synthetic_cpab/lognormal",
    use_signal=SIGNAL_TO_TRAIN   # or "cbfv"
)

loader = DataLoader(
    dataset,
    batch_size=2,     
    shuffle=True,
    collate_fn=collate_fn
)

In [ ]:
batch = next(iter(loader))

x = batch["x"].unsqueeze(1)       # [B,1,L]
x_w = batch["x_w"].unsqueeze(1)

kp = batch["kp"]                  # [B,L]
kp_w = batch["kp_w"]
match_mask = batch["match_mask"]  # [B,L,L]

In [ ]:
model = TimePointModel()
model = model.to(device)

criterion = TimePointOverallLoss(
    mp=1.0,
    mn=0.1,
    lambda_desc=1.0
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# forward 
S, D = model(x)
S_w, D_w = model(x_w)

print("S:", S.shape)
print("D:", D.shape)
print("match_mask:", match_mask.shape)

In [ ]:
import os

save_dir = "../checkpoints"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
epochs = 30

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in loader:
        x = batch["x"].to(device).unsqueeze(1)     # [B,1,L]
        x_w = batch["x_w"].to(device).unsqueeze(1)

        kp = batch["kp"].to(device)
        kp_w = batch["kp_w"].to(device)

        match_mask = batch["match_mask"].to(device)

        # forward 
        S_logits, D = model(x)
        S_w_logits, D_w = model(x_w)

        # loss
        loss, loss_dict = criterion(
            S_logits, kp,
            S_w_logits, kp_w,
            D, D_w,
            match_mask
        )

        # backward 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"\nEpoch {epoch}")
    print(f"Total loss: {total_loss:.4f}")
    print(loss_dict)

    # saving model weights
    save_path = os.path.join(save_dir, f"model_epoch_{epoch}.pth")
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": total_loss,
    }, save_path)

In [ ]:
def visualize(sample, model):
    model.eval()

    x = sample["x"].unsqueeze(0).unsqueeze(0).to(device)
    x_w = sample["x_w"].unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        S_logits, _ = model(x)
        S_w_logits, _ = model(x_w)

    S = torch.sigmoid(S_logits).cpu().numpy()[0]
    S_w = torch.sigmoid(S_w_logits).cpu().numpy()[0]

    t = np.arange(len(S))

    plt.figure(figsize=(12,5))
    plt.plot(t, sample["x"], label=SIGNAL_TO_TRAIN.upper())
    plt.plot(t, sample["x_w"], label="Warped", alpha=0.7)

    plt.scatter(t[S > 0.5], sample["x"][S > 0.5], color="red", label="KP")
    plt.scatter(t[S_w > 0.5], sample["x_w"][S_w > 0.5], color="blue", label="KP warped")

    plt.legend()
    plt.title("Predicted keypoints")
    plt.show()

In [ ]:
sample = dataset[0]
visualize(sample, model)

# Fine-tuning on real data

In [ ]:
model = TimePointModel().to(device)

checkpoint = torch.load("../checkpoints/model_epoch_29.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

Encoder freezing

In [ ]:
for param in model.encoder.parameters():
    param.requires_grad = False

In [ ]:
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

In [ ]:
dataset = NPZLoader(
    "../../data_for_finetuning/baseline",
    use_signal=SIGNAL_TO_TRAIN,
    window=1024
)

In [ ]:
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in loader:
        x = batch["x"].to(device).unsqueeze(1)     # [B,1,L]
        x_w = batch["x_w"].to(device).unsqueeze(1)

        kp = batch["kp"].to(device)
        kp_w = batch["kp_w"].to(device)

        match_mask = batch["match_mask"].to(device)

        # forward 
        S_logits, D = model(x)
        S_w_logits, D_w = model(x_w)

        # loss
        loss, loss_dict = criterion(
            S_logits, kp,
            S_w_logits, kp_w,
            D, D_w,
            match_mask
        )

        # backward 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"\nEpoch {epoch}")
    print(f"Total loss: {total_loss:.4f}")
    print(loss_dict)

    # saving model weights
    save_path = os.path.join(save_dir, f"model_epoch_{epoch}.pth")
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": total_loss,
    }, save_path)

In [ ]:
sample = dataset[0]
visualize(sample, model)

In [ ]:
pth = np.load("/Users/weronikadomczewska/Documents/magisterskie/semestr3/magisterka/data_for_finetuning/baseline/sample_1.npz")

print(pth.files)